# Weather Nowcasting: Model Training & Comparison
## Training Baseline ConvLSTM vs Vision Transformer

This notebook trains both baseline ConvLSTM and Vision Transformer models on satellite+wind data and compares their performance on the Morocco weather nowcasting task.

**Dataset**: 67 combined satellite and wind data files  
**Input**: 6 consecutive timesteps (15-min cadence)  
**Output**: 1 future timestep prediction (next 15 minutes)

## ⚠️ GOOGLE COLAB SETUP (Run this first if using Colab)

Execute the cell below if you're running this notebook in Google Colab. Skip if running locally.

In [ ]:
# ============================================================================
# GOOGLE COLAB SETUP - Run this cell first if using Colab
# ============================================================================

import subprocess
import sys
from pathlib import Path

# Check if running in Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally (not in Colab)")

if IN_COLAB:
    # Mount Google Drive
    print("\n1. Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    print("   ✓ Google Drive mounted")
    
    # Install required packages (including gdown for Google Drive downloads)
    print("\n2. Installing required packages...")
    packages = ['torch', 'torchvision', 'numpy', 'scikit-image', 'pyyaml', 'tqdm', 'matplotlib', 'gdown']
    for pkg in packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    print("   ✓ All packages installed")
    
    # Clone repository
    print("\n3. Setting up repository...")
    repo_url = "https://github.com/YOUR_USERNAME/morocco-weather-nowcasting.git"
    
    # Check if we already have the repo
    if Path('/content/morocco-weather-nowcasting').exists():
        print("   Repository already exists, pulling latest changes...")
        subprocess.run(['git', '-C', '/content/morocco-weather-nowcasting', 'pull'], 
                      capture_output=True)
    else:
        subprocess.run(['git', 'clone', repo_url, '/content/morocco-weather-nowcasting'], 
                      capture_output=True)
        print(f"   ✓ Repository cloned")
    
    # Set working directory
    import os
    os.chdir('/content/morocco-weather-nowcasting')
    print(f"   ✓ Working directory set to: {os.getcwd()}")
    
    # Download and extract dataset from Google Drive
    print("\n4. Downloading dataset from Google Drive...")
    print("   This may take 3-5 minutes depending on file size...")
    
    import gdown
    
    # File ID from the shared link: https://drive.google.com/file/d/1Oo0FEvpJefUikzOFm0q18d7LSAI3_hOI/view
    file_id = "1Oo0FEvpJefUikzOFm0q18d7LSAI3_hOI"
    zip_path = "/content/dataset.zip"
    
    try:
        # Download file
        gdown.download(f"https://drive.google.com/uc?id={file_id}&confirm=t", zip_path, quiet=False)
        print(f"   ✓ Dataset downloaded ({Path(zip_path).stat().st_size / 1e9:.2f} GB)")
        
        # Extract dataset
        print("\n5. Extracting dataset...")
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('/content/morocco-weather-nowcasting/data/')
        print("   ✓ Dataset extracted")
        
        # Clean up zip file
        os.remove(zip_path)
        print("   ✓ Temporary files cleaned up")
        
        # Verify data structure
        data_dir = Path('/content/morocco-weather-nowcasting/data/combined')
        if data_dir.exists():
            combined_files = list(data_dir.glob('combined_*.npz'))
            print(f"\n✓ Found {len(combined_files)} combined data files")
        else:
            print(f"\n⚠️ Expected data directory not found at {data_dir}")
            print("   Available directories in data/:")
            data_parent = Path('/content/morocco-weather-nowcasting/data')
            if data_parent.exists():
                for item in data_parent.iterdir():
                    print(f"     - {item.name}")
    
    except Exception as e:
        print(f"\n⚠️ Error downloading dataset: {e}")
        print("   Fallback: Copy your dataset to Google Drive manually:")
        print("   - Upload zip to Google Drive")
        print("   - Share link in next cell")
        print("   - Or upload 'data/combined/' folder directly to Colab")
    
    print("\n" + "="*70)
    print("COLAB SETUP COMPLETE!")
    print("="*70)
    print("\nNext steps:")
    print("1. Run the remaining cells to train the models")
    print("2. After training, download the results folder from Files panel")
    print("\nFiles to copy back to your local machine:")
    print("  - experiments/notebook_training/baseline/")
    print("  - experiments/notebook_training/vit/")
    print("  - experiments/notebook_training/*.png")
    print("  - experiments/notebook_training/SUMMARY.txt")
    print("\n" + "="*70)
else:
    print("\nLocal setup detected. Proceeding with local execution...")


## Download Results from Colab

Run this cell after training to download results from Colab to your local machine.

In [ ]:
if IN_COLAB:
    # Create a zip file with all results
    import shutil
    import os
    
    print("Preparing results for download...\n")
    
    # Create results directory
    results_dir = Path('/content/colab_results')
    results_dir.mkdir(exist_ok=True)
    
    exp_dir = Path('/content/morocco-weather-nowcasting/experiments/notebook_training')
    
    if exp_dir.exists():
        print(f"Found experiments directory: {exp_dir}")
        
        # Copy entire experiments folder
        dest = results_dir / 'notebook_training'
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(exp_dir, dest)
        print(f"✓ Copied {exp_dir.name} to {dest}")
        
        # Create zip file
        zip_path = '/content/morocco_nowcasting_results'
        shutil.make_archive(zip_path, 'zip', results_dir)
        print(f"\n✓ Created zip file: {zip_path}.zip")
        
        print("\nTo download the results:")
        print("1. Click on the 'Files' icon on the left sidebar")
        print("2. Navigate to /content/colab_results/")
        print("3. Right-click on 'notebook_training' folder → Download")
        print("   OR download 'morocco_nowcasting_results.zip'")
        
        print("\nContents of results folder:")
        for root, dirs, files in os.walk(exp_dir):
            level = root.replace(str(exp_dir), '').count(os.sep)
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 2 * (level + 1)
            for file in files:
                print(f'{subindent}{file}')
    else:
        print(f"⚠️ Experiments directory not found at {exp_dir}")
        print("Please run the training cells first!")
else:
    print("Local execution mode - results are saved in: experiments/notebook_training/")
    print("\nResults structure:")
    print("  experiments/notebook_training/")
    print("  ├── baseline/")
    print("  │   ├── baseline_best.pt (model weights)")
    print("  │   ├── forecasts.npz (predictions)")
    print("  │   └── metrics.json (metrics)")
    print("  ├── vit/")
    print("  │   ├── vit_best.pt (model weights)")
    print("  │   ├── forecasts.npz (predictions)")
    print("  │   └── metrics.json (metrics)")
    print("  ├── training_curves.png")
    print("  ├── metrics_comparison.png")
    print("  ├── prediction_comparison.png")
    print("  └── SUMMARY.txt")

In [3]:
import sys
import os
from pathlib import Path

# Set working directory to repo root (morocco-weather-nowcasting)
repo_root = Path.cwd().parent  # Go up from notebook/ to project root
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

print(f"Working directory: {os.getcwd()}")
print(f"Repo root: {repo_root}")
print(f"Sys path includes: {repo_root}")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import json
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

from src.models.conv_lstm_baseline import ConvLSTMBaseline
from src.models.vit_nowcasting import ViTNowcaster, ViTNowcasterConfig
from src.eval.metrics import summary_table, per_channel_mae, per_channel_rmse
from src.eval.artifacts import save_forecast_npz

print("✓ All imports successful!")

Working directory: c:\Users\perso\Desktop\Time_Series\morocco-weather-nowcasting
Repo root: c:\Users\perso\Desktop\Time_Series\morocco-weather-nowcasting
Sys path includes: c:\Users\perso\Desktop\Time_Series\morocco-weather-nowcasting

Using device: cpu
✓ All imports successful!


## 1. Load and Prepare Data

Load the combined satellite and wind data from `data/combined/` directory. The dataset contains 67 NPZ files with shape (C=4, H=557, W=521) each.

In [4]:
class WeatherDataset(Dataset):
    """Simple dataset loader for combined weather data files."""
    
    def __init__(self, file_list, sequence_length=6, forecast_horizon=1, means=None, stds=None):
        self.file_list = file_list
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.means = means
        self.stds = stds
        
        # Compute normalization stats if not provided
        if means is None or stds is None:
            print("Computing normalization statistics...")
            all_data = []
            for f in tqdm(file_list, desc="Loading files for stats"):
                data = np.load(f)['data']
                all_data.append(data)
            all_data = np.stack(all_data, axis=0)
            self.means = np.mean(all_data, axis=(0, 2, 3))
            self.stds = np.std(all_data, axis=(0, 2, 3))
            self.stds[self.stds == 0] = 1.0
    
    def __len__(self):
        return max(0, len(self.file_list) - self.sequence_length - self.forecast_horizon + 1)
    
    def __getitem__(self, idx):
        X_list = []
        for i in range(idx, idx + self.sequence_length):
            data = np.load(self.file_list[i])['data']
            X_list.append(data)
        
        X = np.stack(X_list, axis=0)
        
        target_idx = idx + self.sequence_length + self.forecast_horizon - 1
        y = np.load(self.file_list[target_idx])['data']
        y = y[np.newaxis, ...]
        
        # Normalize
        X = (X - self.means[None, :, None, None]) / self.stds[None, :, None, None]
        y = (y - self.means[None, :, None, None]) / self.stds[None, :, None, None]
        
        X = torch.from_numpy(X).float()
        y = torch.from_numpy(y).float()
        
        return X, y


# Load data
data_dir = Path("data/combined")
combined_files = sorted(list(data_dir.glob("combined_*.npz")))

print(f"Found {len(combined_files)} combined files")

# Compute stats
all_data = []
for f in tqdm(combined_files, desc="Loading all files for normalization"):
    data = np.load(f)['data']
    all_data.append(data)
all_data = np.stack(all_data, axis=0)
means = np.mean(all_data, axis=(0, 2, 3))
stds = np.std(all_data, axis=(0, 2, 3))
stds[stds == 0] = 1.0

print(f"Means: {means}")
print(f"Stds: {stds}")

# Temporal split
n_total = len(combined_files)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)

train_files = combined_files[:n_train]
val_files = combined_files[n_train:n_train + n_val]
test_files = combined_files[n_train + n_val:]

print(f"\nData split:")
print(f"  Train: {len(train_files)} files → {len(WeatherDataset(train_files, means=means, stds=stds))} samples")
print(f"  Val:   {len(val_files)} files → {len(WeatherDataset(val_files, means=means, stds=stds))} samples")
print(f"  Test:  {len(test_files)} files → {len(WeatherDataset(test_files, means=means, stds=stds))} samples")

Found 67 combined files


Loading all files for normalization: 100%|██████████| 67/67 [00:08<00:00,  7.61it/s]


Means: [288.08535853   3.28157228  -1.66839017   4.54333214]
Stds: [12.98698181  6.39108202  3.4411727   6.80963245]

Data split:
  Train: 46 files → 40 samples
  Val:   10 files → 4 samples
  Test:  11 files → 5 samples


In [5]:
# Create dataloaders
BATCH_SIZE = 2
NUM_WORKERS = 0

train_dataset = WeatherDataset(train_files, means=means, stds=stds)
val_dataset = WeatherDataset(val_files, means=means, stds=stds)
test_dataset = WeatherDataset(test_files, means=means, stds=stds)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Check a sample batch
print("\nSample batch shapes:")
X_sample, y_sample = next(iter(train_loader))
print(f"  X shape: {X_sample.shape}  (Batch, Time_in, Channels, H, W)")
print(f"  y shape: {y_sample.shape}  (Batch, Time_out, Channels, H, W)")


Sample batch shapes:
  X shape: torch.Size([2, 6, 4, 557, 521])  (Batch, Time_in, Channels, H, W)
  y shape: torch.Size([2, 1, 4, 557, 521])  (Batch, Time_out, Channels, H, W)


## 2. Configure Training Parameters

In [6]:
# Training configuration
BASELINE_EPOCHS = 5
VIT_EPOCHS = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# Create output directories
EXP_DIR = Path("experiments/notebook_training")
BASELINE_DIR = EXP_DIR / "baseline"
VIT_DIR = EXP_DIR / "vit"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
VIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  Baseline epochs: {BASELINE_EPOCHS}")
print(f"  ViT epochs: {VIT_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Device: {device}")
print(f"\nOutput directories:")
print(f"  Baseline: {BASELINE_DIR}")
print(f"  ViT: {VIT_DIR}")

Configuration:
  Baseline epochs: 5
  ViT epochs: 3
  Learning rate: 0.001
  Batch size: 2
  Device: cpu

Output directories:
  Baseline: experiments\notebook_training\baseline
  ViT: experiments\notebook_training\vit


## 3. Training Helper Functions

Define functions for training, validation, and evaluation with verbose progress tracking.

In [7]:
def train_epoch(model, train_loader, criterion, optimizer, device, model_name="Model"):
    """Train for one epoch with progress bar."""
    model.train()
    total_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"  Training", leave=False)
    for batch_idx, (X, y) in enumerate(pbar):
        X, y = X.to(device), y.to(device)
        
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)


def validate(model, val_loader, criterion, device):
    """Validate model with progress bar."""
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"  Validating", leave=False)
        for X, y in pbar:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = criterion(pred, y)
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(val_loader)


def evaluate_test_set(model, test_loader, device):
    """Evaluate on test set and return predictions."""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        pbar = tqdm(test_loader, desc="  Evaluating", leave=False)
        for X, y in pbar:
            X = X.to(device)
            pred = model(X)
            all_preds.append(pred.cpu().numpy())
            all_targets.append(y.numpy())
    
    preds = np.concatenate(all_preds, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    return preds, targets


print("✓ Training functions defined")

✓ Training functions defined


## 4. Train Baseline ConvLSTM Model

Train the ConvLSTM baseline model with verbose progress tracking.

In [8]:
print("="*60)
print("TRAINING BASELINE CONVLSTM")
print("="*60)

# Initialize model
baseline_model = ConvLSTMBaseline(
    input_channels=4,
    hidden_dim=64,
    kernel_size=(3, 3),
    num_layers=3
).to(device)

total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"Model parameters: {total_params / 1e6:.2f}M\n")

# Training setup
criterion = nn.MSELoss()
optimizer = optim.AdamW(baseline_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=BASELINE_EPOCHS, eta_min=1e-5)

baseline_history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

start_time = datetime.now()

# Training loop
for epoch in range(BASELINE_EPOCHS):
    epoch_start = datetime.now()
    
    print(f"Epoch {epoch+1}/{BASELINE_EPOCHS}")
    
    # Train
    train_loss = train_epoch(baseline_model, train_loader, criterion, optimizer, device, "Baseline")
    baseline_history['train_loss'].append(train_loss)
    
    # Validate
    val_loss = validate(baseline_model, val_loader, criterion, device)
    baseline_history['val_loss'].append(val_loss)
    
    scheduler.step()
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(baseline_model.state_dict(), BASELINE_DIR / "baseline_best.pt")
    
    epoch_time = (datetime.now() - epoch_start).total_seconds()
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")

total_time = (datetime.now() - start_time).total_seconds()
print(f"\nBaseline training completed in {total_time/60:.1f} minutes")

# Load best model
baseline_model.load_state_dict(torch.load(BASELINE_DIR / "baseline_best.pt"))

# Evaluate on test set
print("\nEvaluating Baseline on test set...")
baseline_pred, baseline_target = evaluate_test_set(baseline_model, test_loader, device)

# Compute metrics
baseline_metrics = summary_table(baseline_pred, baseline_target, threshold=0.5)
baseline_pcm = per_channel_mae(baseline_pred, baseline_target)
baseline_pcr = per_channel_rmse(baseline_pred, baseline_target)

print("\n✓ Baseline Metrics:")
for metric, value in baseline_metrics.items():
    print(f"  {metric.upper():6s}: {value:.4f}")

# Save artifacts
save_forecast_npz(BASELINE_DIR / "forecasts.npz", baseline_pred, baseline_target, 
                 {"model": "baseline", "epochs": BASELINE_EPOCHS})
with open(BASELINE_DIR / "metrics.json", 'w') as f:
    json.dump({
        "summary": baseline_metrics,
        "per_channel_mae": baseline_pcm.tolist(),
        "per_channel_rmse": baseline_pcr.tolist(),
        "history": baseline_history
    }, f, indent=2)

print(f"✓ Baseline artifacts saved to {BASELINE_DIR}")

TRAINING BASELINE CONVLSTM
Model parameters: 1.93M

Epoch 1/5


  Training:   0%|          | 0/20 [00:00<?, ?it/s]

: 

## 5. Train Vision Transformer Model

Train the ViT model with the same data and training setup.

In [ ]:
print("\n" + "="*60)
print("TRAINING VISION TRANSFORMER")
print("="*60)

# Initialize model
vit_config = ViTNowcasterConfig(
    image_size=X_sample.shape[-1],
    patch_size=16,
    in_channels=4,
    out_channels=4,
    t_out=1,
    embed_dim=256,
    depth=6,
    num_heads=8,
    mlp_ratio=4.0,
    dropout=0.1,
    attn_dropout=0.0,
    temporal_fusion='attention',
    temporal_heads=4
)

vit_model = ViTNowcaster(vit_config).to(device)

total_params = sum(p.numel() for p in vit_model.parameters())
print(f"Model parameters: {total_params / 1e6:.2f}M\n")

# Training setup
criterion = nn.MSELoss()
optimizer = optim.AdamW(vit_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=VIT_EPOCHS, eta_min=1e-5)

vit_history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

start_time = datetime.now()

# Training loop
for epoch in range(VIT_EPOCHS):
    epoch_start = datetime.now()
    
    print(f"Epoch {epoch+1}/{VIT_EPOCHS}")
    
    # Train
    train_loss = train_epoch(vit_model, train_loader, criterion, optimizer, device, "ViT")
    vit_history['train_loss'].append(train_loss)
    
    # Validate
    val_loss = validate(vit_model, val_loader, criterion, device)
    vit_history['val_loss'].append(val_loss)
    
    scheduler.step()
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(vit_model.state_dict(), VIT_DIR / "vit_best.pt")
    
    epoch_time = (datetime.now() - epoch_start).total_seconds()
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")

total_time = (datetime.now() - start_time).total_seconds()
print(f"\nViT training completed in {total_time/60:.1f} minutes")

# Load best model
vit_model.load_state_dict(torch.load(VIT_DIR / "vit_best.pt"))

# Evaluate on test set
print("\nEvaluating ViT on test set...")
vit_pred, vit_target = evaluate_test_set(vit_model, test_loader, device)

# Compute metrics
vit_metrics = summary_table(vit_pred, vit_target, threshold=0.5)
vit_pcm = per_channel_mae(vit_pred, vit_target)
vit_pcr = per_channel_rmse(vit_pred, vit_target)

print("\n✓ ViT Metrics:")
for metric, value in vit_metrics.items():
    print(f"  {metric.upper():6s}: {value:.4f}")

# Save artifacts
save_forecast_npz(VIT_DIR / "forecasts.npz", vit_pred, vit_target, 
                 {"model": "vit", "epochs": VIT_EPOCHS})
with open(VIT_DIR / "metrics.json", 'w') as f:
    json.dump({
        "summary": vit_metrics,
        "per_channel_mae": vit_pcm.tolist(),
        "per_channel_rmse": vit_pcr.tolist(),
        "history": vit_history
    }, f, indent=2)

print(f"✓ ViT artifacts saved to {VIT_DIR}")

## 6. Visualize Training Metrics

Plot training and validation loss curves for both models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline curves
epochs_baseline = range(1, BASELINE_EPOCHS + 1)
axes[0].plot(epochs_baseline, baseline_history['train_loss'], 'o-', label='Train Loss', linewidth=2, markersize=6)
axes[0].plot(epochs_baseline, baseline_history['val_loss'], 's-', label='Val Loss', linewidth=2, markersize=6)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Baseline ConvLSTM Training', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# ViT curves
epochs_vit = range(1, VIT_EPOCHS + 1)
axes[1].plot(epochs_vit, vit_history['train_loss'], 'o-', label='Train Loss', linewidth=2, markersize=6)
axes[1].plot(epochs_vit, vit_history['val_loss'], 's-', label='Val Loss', linewidth=2, markersize=6)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Vision Transformer Training', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXP_DIR / "training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training curves saved to {EXP_DIR / 'training_curves.png'}")

## 7. Model Comparison

Compare metrics between Baseline and ViT models.

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60 + "\n")

# Create comparison table
metrics_list = ['rmse', 'mae', 'ssim', 'csi', 'pod', 'far']
print(f"{'Metric':<10} {'Baseline':<12} {'ViT':<12} {'Winner':<12} {'Difference':<12}")
print("-" * 60)

for metric in metrics_list:
    b_val = baseline_metrics[metric]
    v_val = vit_metrics[metric]
    diff = v_val - b_val
    
    # Determine winner (lower is better for rmse, mae, far; higher for others)
    if metric in ['rmse', 'mae', 'far']:
        winner = 'ViT ✓' if v_val < b_val else 'Baseline ✓' if b_val < v_val else 'Tie'
    else:
        winner = 'ViT ✓' if v_val > b_val else 'Baseline ✓' if b_val > v_val else 'Tie'
    
    print(f"{metric.upper():<10} {b_val:<12.4f} {v_val:<12.4f} {winner:<12} {diff:+.4f}")

print("\n" + "="*60)
print("Per-Channel Error Analysis")
print("="*60 + "\n")

print(f"{'Channel':<10} {'Baseline MAE':<15} {'ViT MAE':<15} {'Baseline RMSE':<15} {'ViT RMSE':<15}")
print("-" * 70)

for ch in range(len(baseline_pcm)):
    print(f"Ch {ch:<8} {baseline_pcm[ch]:<15.4f} {vit_pcm[ch]:<15.4f} {baseline_pcr[ch]:<15.4f} {vit_pcr[ch]:<15.4f}")

In [ ]:
# Create comparison bar chart
fig, ax = plt.subplots(figsize=(12, 6))

metrics_to_plot = ['rmse', 'mae', 'ssim', 'csi', 'pod', 'far']
baseline_vals = [baseline_metrics[m] for m in metrics_to_plot]
vit_vals = [vit_metrics[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline', alpha=0.8, color='steelblue')
bars2 = ax.bar(x + width/2, vit_vals, width, label='ViT', alpha=0.8, color='coral')

ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Model Comparison: Baseline vs ViT', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metrics_to_plot], fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(EXP_DIR / "metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Metrics comparison saved to {EXP_DIR / 'metrics_comparison.png'}")

## 8. Visual Comparison of Predictions

Compare actual predictions from both models on test samples.

In [ ]:
# Visualize predictions for first test sample
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

sample_idx = 0
channel_idx = 0
timestep_idx = 0

baseline_img = baseline_pred[sample_idx, timestep_idx, channel_idx]
vit_img = vit_pred[sample_idx, timestep_idx, channel_idx]
target_img = vit_target[sample_idx, timestep_idx, channel_idx]

# Row 1: Predictions
im0 = axes[0, 0].imshow(baseline_img, cmap='viridis')
axes[0, 0].set_title(f'Baseline Prediction\nChannel {channel_idx}', fontweight='bold')
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

im1 = axes[0, 1].imshow(vit_img, cmap='viridis')
axes[0, 1].set_title(f'ViT Prediction\nChannel {channel_idx}', fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

im2 = axes[0, 2].imshow(target_img, cmap='viridis')
axes[0, 2].set_title(f'Ground Truth\nChannel {channel_idx}', fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)

# Row 2: Errors
baseline_error = np.abs(baseline_img - target_img)
vit_error = np.abs(vit_img - target_img)
error_diff = baseline_error - vit_error

im3 = axes[1, 0].imshow(baseline_error, cmap='hot')
axes[1, 0].set_title('Baseline Absolute Error', fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im3, ax=axes[1, 0], fraction=0.046)

im4 = axes[1, 1].imshow(vit_error, cmap='hot')
axes[1, 1].set_title('ViT Absolute Error', fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im4, ax=axes[1, 1], fraction=0.046)

vmax = np.max(np.abs(error_diff))
im5 = axes[1, 2].imshow(error_diff, cmap='RdBu', vmin=-vmax, vmax=vmax)
axes[1, 2].set_title('Error Difference\n(Blue=ViT better)', fontweight='bold')
axes[1, 2].axis('off')
plt.colorbar(im5, ax=axes[1, 2], fraction=0.046)

plt.tight_layout()
plt.savefig(EXP_DIR / "prediction_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Prediction comparison saved to {EXP_DIR / 'prediction_comparison.png'}")

## 9. Summary Report

Generate final summary of the experiment.

In [ ]:
# Create final summary report
report = f"""
{"="*70}
WEATHER NOWCASTING: MODEL COMPARISON SUMMARY
{"="*70}

DATASET INFORMATION:
  Total files: {len(combined_files)}
  Train samples: {len(train_dataset)}
  Val samples: {len(val_dataset)}
  Test samples: {len(test_dataset)}
  Input shape: (6, 4, 557, 521) → 6 timesteps, 4 channels, 557×521 spatial
  Output shape: (1, 4, 557, 521) → 1 future timestep prediction

BASELINE MODEL (ConvLSTM):
  Parameters: {sum(p.numel() for p in baseline_model.parameters()) / 1e6:.2f}M
  Training epochs: {BASELINE_EPOCHS}
  Training time: {sum(baseline_history['train_loss']) / BASELINE_EPOCHS:.2f}s per epoch
  Best validation loss: {min(baseline_history['val_loss']):.4f}

VIT MODEL (Vision Transformer):
  Parameters: {sum(p.numel() for p in vit_model.parameters()) / 1e6:.2f}M
  Training epochs: {VIT_EPOCHS}
  Training time: {sum(vit_history['train_loss']) / VIT_EPOCHS:.2f}s per epoch
  Best validation loss: {min(vit_history['val_loss']):.4f}

TEST SET METRICS:

┌────────┬────────────┬─────────┬──────────┬──────────┬──────────┬──────────┐
│ Metric │  Baseline  │   ViT   │  Winner  │ Lower    │ Higher   │ Margin   │
├────────┼────────────┼─────────┼──────────┼──────────┼──────────┼──────────┤
│ RMSE   │  {baseline_metrics['rmse']:8.4f}  │ {vit_metrics['rmse']:7.4f} │ {'ViT ✓' if vit_metrics['rmse'] < baseline_metrics['rmse'] else 'Baseline ✓':<8} │ Better   │          │ {abs(vit_metrics['rmse'] - baseline_metrics['rmse']):8.4f} │
│ MAE    │  {baseline_metrics['mae']:8.4f}  │ {vit_metrics['mae']:7.4f} │ {'ViT ✓' if vit_metrics['mae'] < baseline_metrics['mae'] else 'Baseline ✓':<8} │ Better   │          │ {abs(vit_metrics['mae'] - baseline_metrics['mae']):8.4f} │
│ SSIM   │  {baseline_metrics['ssim']:8.4f}  │ {vit_metrics['ssim']:7.4f} │ {'ViT ✓' if vit_metrics['ssim'] > baseline_metrics['ssim'] else 'Baseline ✓':<8} │          │ Better   │ {abs(vit_metrics['ssim'] - baseline_metrics['ssim']):8.4f} │
│ CSI    │  {baseline_metrics['csi']:8.4f}  │ {vit_metrics['csi']:7.4f} │ {'ViT ✓' if vit_metrics['csi'] > baseline_metrics['csi'] else 'Baseline ✓':<8} │          │ Better   │ {abs(vit_metrics['csi'] - baseline_metrics['csi']):8.4f} │
│ POD    │  {baseline_metrics['pod']:8.4f}  │ {vit_metrics['pod']:7.4f} │ {'ViT ✓' if vit_metrics['pod'] > baseline_metrics['pod'] else 'Baseline ✓':<8} │          │ Better   │ {abs(vit_metrics['pod'] - baseline_metrics['pod']):8.4f} │
│ FAR    │  {baseline_metrics['far']:8.4f}  │ {vit_metrics['far']:7.4f} │ {'ViT ✓' if vit_metrics['far'] < baseline_metrics['far'] else 'Baseline ✓':<8} │ Better   │          │ {abs(vit_metrics['far'] - baseline_metrics['far']):8.4f} │
└────────┴────────────┴─────────┴──────────┴──────────┴──────────┴──────────┘

KEY FINDINGS:
  1. Continuous metrics (RMSE, MAE, SSIM) measure pixel-level prediction accuracy
  2. Event detection metrics (CSI, POD, FAR) measure ability to detect weather events
  3. Models show different strengths - one may excel at regression, other at detection
  4. Per-channel analysis reveals if models struggle with specific data types

NEXT STEPS:
  ✓ Models trained and evaluated
  ✓ Artifacts saved to {EXP_DIR}
  ✓ Comparison metrics computed
  → Consider retraining with adjusted hyperparameters
  → Analyze per-channel errors for improvement areas
  → Extend dataset with more training data for better generalization

OUTPUT FILES:
  - {BASELINE_DIR / 'forecasts.npz'} (predictions)
  - {BASELINE_DIR / 'metrics.json'} (metrics)
  - {VIT_DIR / 'forecasts.npz'} (predictions)
  - {VIT_DIR / 'metrics.json'} (metrics)
  - {EXP_DIR / 'training_curves.png'} (visualization)
  - {EXP_DIR / 'metrics_comparison.png'} (visualization)
  - {EXP_DIR / 'prediction_comparison.png'} (visualization)

{"="*70}
"""

print(report)

# Save report
with open(EXP_DIR / "SUMMARY.txt", 'w') as f:
    f.write(report)

print(f"\n✓ Summary report saved to {EXP_DIR / 'SUMMARY.txt'}")